#  Gamma/neutron discrimination based on ML

- Workflow based on R. S. Molina, I. R. Morales, M. L. Crespo, V. G. Costa, S. Carrato and G. Ramponi, "An End-to-End Workflow to Efficiently Compress and Deploy DNN Classifiers on SoC/FPGA", in IEEE Embedded Systems Letters, vol. 16, no. 3, pp. 255-258, Sept. 2024, doi: 10.1109/LES.2023.3343030.

- Code adapted from the official repository of "An End-to-End Workflow to Efficiently Compress and Deploy DNN Classifiers on SoC/FPGA".

- Using open dataset from: https://doi.org/10.5281/zenodo.8037058 .

- hls4ml documentation: https://fastmachinelearning.org/hls4ml/ .

### Import libraries

In [1]:
import os
import numpy as np
import tensorflow as tf 
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from qkeras import *
from qkeras import QActivation
from qkeras import QDense, QConv2DBatchnorm
import hls4ml

###  Path to Vitis HLS

Vitis HLS directory should be specified to use hls4ml with the high-level synthesis tool.

In [2]:
# PATH Xilinx Virtual Machine
os.environ['PATH'] = '/mnt/d/Xilinx/Vitis_HLS/2022.2/bin:' + os.environ['PATH']
os.environ['PATH'] = '/mnt/d/Xilinx/Vitis/2022.2/bin:' + os.environ['PATH']
os.environ['PATH'] = '/mnt/d/Xilinx/Vivado/2022.2/bin:' + os.environ['PATH']

### Load model

In [3]:
# Load keras model 

from qkeras.utils import _add_supported_quantized_objects
co = {}
_add_supported_quantized_objects(co)

model = load_model('./models/studentModel_GN_smr4078.h5', custom_objects=co)
model.summary()

Model: "studentMLP"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 fc1 (QDense)                (None, 6)                 972       
                                                                 
 relu0 (QActivation)         (None, 6)                 0         
                                                                 
 dropout_9 (Dropout)         (None, 6)                 0         
                                                                 
 fc2 (QDense)                (None, 4)                 28        
                                                                 
 relu1 (QActivation)         (None, 4)                 0         
                                                                 
 dropout_10 (Dropout)        (None, 4)                 0         
                                                                 
 fc3 (QDense)                (None, 2)                 1

###  hls4ml integration

hls4ml is a Python package developed for converting machine learning (ML) models into HLS (High-Level Synthesis) projects, enabling deployment of ML-based inference on hardware like FPGAs. More details can be found at hls4ml documentation.

The user can control several options related to the model, including:

- Precision: Define the precision of the calculations in your model (e.g., fixed-point or floating-point representation).

- Dataflow/Resource Reuse: Control the level of parallelism or streaming in model implementations, with varying degrees of pipelining.

- Quantization Aware Training: Achieve optimized performance at low precision by using tools like QKeras. Models trained with QKeras benefit automatically from hls4ml's parsing of QKeras models during inference.

An HLS configuration should be created using the function hls4ml.utils.config_from_keras_model(kerasModel, granularity), where kerasModel is the pre-trained model you want to implement on an FPGA, and granularity determines the configuration level. The two possible values for granularity are:

- 'model': The same configuration applies to the entire model (e.g., all layers use 16-bit fixed-point precision).

- 'name': Layer-specific configurations can be applied (e.g., the input layer can be defined in 8-bit fixed-point precision, while the second layer is set to 16-bit fixed-point precision).

In [4]:
hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

Interpreting Sequential
Topology:
Layer name: inputLayer, layer type: InputLayer, input shapes: [[None, 161]], output shape: [None, 161]
Layer name: fc1, layer type: QDense, input shapes: [[None, 161]], output shape: [None, 6]
Layer name: relu0, layer type: Activation, input shapes: [[None, 6]], output shape: [None, 6]
Layer name: fc2, layer type: QDense, input shapes: [[None, 6]], output shape: [None, 4]
Layer name: relu1, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc3, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 2]
Layer name: relu2, layer type: Activation, input shapes: [[None, 2]], output shape: [None, 2]
Layer name: fc4, layer type: QDense, input shapes: [[None, 2]], output shape: [None, 4]
Layer name: relu3, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc5, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 3]
Layer name: relu4, layer type: Activation, in

In [5]:
from src import plotting

print("-----------------------------------")
plotting.print_dict(hls_config)
print("-----------------------------------")

-----------------------------------
Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  inputLayer
    Trace:           False
    Precision
      result:        auto
  fc1
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<8,5,TRN,WRAP,0>
      bias:          fixed<8,5,TRN,WRAP,0>
  fc1_linear
    Trace:           False
    Precision
      result:        auto
  relu0
    Trace:           False
    Precision
      result:        fixed<8,1,RND_CONV,SAT,0>
  fc2
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<8,5,TRN,WRAP,0>
      bias:          fixed<8,5,TRN,WRAP,0>
  fc2_linear
    Trace:           False
    Precision
      result:        auto
  relu1
    Trace:           False
    Precision
      result:        fixed<8,1,RND_CONV,SAT,0>
  fc3
    Trace:           False
    Preci

In [6]:
for Layer in hls_config['LayerName'].keys():
    hls_config['LayerName'][Layer]['Strategy'] = 'Latency'
    hls_config['LayerName'][Layer]['ReuseFactor'] = 1
    hls_config['LayerName'][Layer]['Precision'] = 'ap_fixed<8,4>'

hls_config['LayerName']['outputActivation']['Strategy'] = 'Stable'
hls_config['Model']['Precision'] = 'ap_fixed<16,6>'

hls_config['LayerName']['inputLayer']['Precision'] = 'ap_fixed<16,6>'

###  hls4ml with Vitis HLS as backend

In [ ]:
# Create configuration for Vitis HLS as backend.
cfg = hls4ml.converters.create_config(backend='Vitis')

# HLSConfig correspond to the configuration created in hls_config 
cfg['HLSConfig']  = hls_config
# Model to be converted
cfg['KerasModel'] = model
# Folder where the HLS project will be created
cfg['OutputDir']  = './hlsPrj/'
# FPGA part 
cfg['Part'] = 'xc7z020clg484-1'  # PYNQ-Z1 or Zedboard: xc7z020clg484-1  
  
hls_model = hls4ml.converters.keras_to_hls(cfg)

hls_model.compile()
### Usar en terminal el sigueinte comando
# $env:PATH += ";D:\Xilinx\Vitis_HLS\2022.2\bin"
# vitis_hls -version
# cd .\pythonModel\hlsPrj\
# vitis_hls -f .\build_prj.tcl


Interpreting Sequential
Topology:
Layer name: inputLayer, layer type: InputLayer, input shapes: [[None, 161]], output shape: [None, 161]
Layer name: fc1, layer type: QDense, input shapes: [[None, 161]], output shape: [None, 6]
Layer name: relu0, layer type: Activation, input shapes: [[None, 6]], output shape: [None, 6]
Layer name: fc2, layer type: QDense, input shapes: [[None, 6]], output shape: [None, 4]
Layer name: relu1, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc3, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 2]
Layer name: relu2, layer type: Activation, input shapes: [[None, 2]], output shape: [None, 2]
Layer name: fc4, layer type: QDense, input shapes: [[None, 2]], output shape: [None, 4]
Layer name: relu3, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc5, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 3]
Layer name: relu4, layer type: Activation, in

Done
"." no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.



Exception: Failed to compile project "myproject"

In [10]:
 # This will perform the synthesis of the HLS project, showing the main results (latency and resource usage) once the process is completed. 

hls_model.build(csim=False, export=False)

Vivado synthesis report not found.
Implementation report not found.
Timing report not found.


{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.554',
  'BestLatency': '41',
  'WorstLatency': '41',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'BRAM_18K': '1',
  'DSP': '339',
  'FF': '36779',
  'LUT': '10746',
  'URAM': '0',
  'AvailableBRAM_18K': '280',
  'AvailableDSP': '220',
  'AvailableFF': '106400',
  'AvailableLUT': '53200',
  'AvailableURAM': '0'},
 'CosimReport': {'RTL': 'Verilog',
  'Status': 'Pass',
  'LatencyMin': 41,
  'LatencyMax': 41,
  'IntervalMin': 1,
  'IntervalMax': 1,
  'LatencyAvg': 41.0,
  'IntervalAvg': 1.0}}